# **Causal scrubbing, illusions and MDL probing: practice**

Notebook for the module "Evaluating interpretability and scientific rigor". It collects the toy runs of the module's lessons, and the numbers in the lesson steps come from its output:

* lesson "Faithfulness: is the interpretation correct?": causal scrubbing of three hypotheses about a model with a known mechanism, and the fraction of loss recovered $R(H)$;
* lesson "Interpretability illusions": a direction that looks like a topic detector inside a corpus (as in Bolukbasi et al.), and a surrogate of an attention head that is faithful on its own distribution and wrong outside it (as in Friedman et al.);
* lesson "Pitfalls of probing": accuracy and online code length (Voita and Titov) of a linear probe and an MLP probe on the real task and on a control task (Hewitt and Liang).

A CPU is enough: on three CPU threads our run of the whole notebook took 7.5 minutes, almost all of it spent training probes in section 3. Every cell sets its seeds, so the numbers repeat on a rerun. The exercises are numbered as in the steps of the lesson "Causal scrubbing, illusions and MDL probing: Practice".

## **Setup**
Only torch and numpy are needed, and Colab already has them. Everything runs on the CPU, so the numbers match the lesson steps on a GPU machine as well.

In [ ]:
import math, json, time
import numpy as np
import torch
import torch.nn.functional as F

R = {}  # all numbers of the notebook; saved to toys_32_results.json at the end
T0 = time.time()
print("torch", torch.__version__)

## **1. Causal scrubbing on a model with a known mechanism**

The input is four features $x_1, x_2, x_3, x_4 \in \{-1, +1\}$; the label is $y = 1$ when the signs of $x_1$ and $x_2$ agree. The features $x_1, x_2, x_4$ are independent and equiprobable, and $x_3$ equals the product $x_1 x_2$ with probability 0.8. The model is not trained; its logit is given by $z = 3\,x_1 x_2 + 1.5\,x_3$; the behavior is the mean binary cross-entropy against the label $y$, in nats.

A hypothesis declares some features unimportant; their values are taken from another random example of the same sample (one random permutation of rows per hypothesis), and the important features stay as they are. The fraction of loss recovered, after Chan et al.:

$$R(H) = \frac{L_{\mathrm{base}} - L_{\mathrm{scrub}}}{L_{\mathrm{base}} - L_{\mathrm{orig}}},$$

where $L_{\mathrm{base}}$ is the loss of the original model against labels shuffled by a random permutation. $N = 40{,}000$ examples, seed 0.

In [ ]:
torch.manual_seed(0)
N = 40000
x = torch.randint(0, 2, (N, 4)).float() * 2 - 1
agree = torch.rand(N) < 0.8
x[:, 2] = torch.where(agree, x[:, 0] * x[:, 1], -x[:, 0] * x[:, 1])
y = (x[:, 0] * x[:, 1] > 0).float()

logit = lambda x: 3 * x[:, 0] * x[:, 1] + 1.5 * x[:, 2]
bce = lambda z, y: F.binary_cross_entropy_with_logits(z, y).item()
accuracy = lambda z, y: ((z > 0).float() == y).float().mean().item()

L_orig = bce(logit(x), y)
L_base = bce(logit(x), y[torch.randperm(N)])
R["scrub"] = {"L_orig": L_orig, "acc_orig": accuracy(logit(x), y), "L_base": L_base,
              "share_x3_eq_x1x2": (x[:, 2] == x[:, 0] * x[:, 1]).float().mean().item(), "share_y1": y.mean().item()}
print(f"L_orig = {L_orig:.3f}, accuracy of the original model {R['scrub']['acc_orig']:.3f}, L_base = {L_base:.3f}")
print(f"share of x3 = x1*x2: {R['scrub']['share_x3_eq_x1x2']:.3f}, share of y = 1: {R['scrub']['share_y1']:.3f}")

HYPOTHESES = {"A": [0, 1], "B": [0, 2], "C": [0, 1, 2]}  # indices of the important features
print(f"{'hypothesis':<11} {'L_scrub':>8} {'accuracy':>9} {'R(H)':>6}")
for name, important in HYPOTHESES.items():
    xs = x[torch.randperm(N)].clone()
    xs[:, important] = x[:, important]
    z = logit(xs)
    L_scrub = bce(z, y)
    R["scrub"][name] = {"L_scrub": L_scrub, "acc": accuracy(z, y), "R": (L_base - L_scrub) / (L_base - L_orig)}
    print(f"{name:<11} {L_scrub:>8.3f} {R['scrub'][name]['acc']:>9.3f} {R['scrub'][name]['R']:>6.3f}")

**Exercise 1. What fraction of loss $R(H)$ does hypothesis B, which declares $x_2$ and $x_4$ unimportant, recover? Give the answer with three decimals.**

Hypothesis B drops accuracy almost to chance, yet its $R$ is not 0: $x_3$ agrees with $x_1 x_2$ in 80% of examples and partly lowers the loss.

## **2. Two interpretability illusions**

**Part 1, as in Bolukbasi et al.** Two corpora of 10,000 points in $\mathbb R^{16}$: the center of corpus A is $+4e_0$, of corpus B is $-4e_0$. Each corpus has 5 topics of 2000 points: the topic center lies 2.5 away from the corpus center in a random direction, with noise of standard deviation 0.5. We take 100 random unit directions $v$, and for each the 10 points of a corpus with the largest projection on $v$. The purity of a set is the share of the most frequent topic among the 10. For comparison, the mean purity of 10 random points of corpus A over 1000 draws. Seed 0.

In [ ]:
torch.manual_seed(0)
D = 16

def corpus(sign):
    center = torch.zeros(D); center[0] = 4.0 * sign
    points, topics = [], []
    for k in range(5):
        u = torch.randn(D); u /= u.norm()
        points.append(center + 2.5 * u + 0.5 * torch.randn(2000, D)); topics.append(torch.full((2000,), k + 1))
    return torch.cat(points), torch.cat(topics)

XA, TA = corpus(+1)
XB, TB = corpus(-1)
V = torch.randn(100, D); V /= V.norm(dim=1, keepdim=True)

def purity(topics, idx):
    counts = torch.bincount(topics[idx], minlength=6)
    return counts.max().item() / len(idx), counts.argmax().item(), counts.max().item()

pA = [purity(TA, (XA @ v).topk(10).indices) for v in V]
pB = [purity(TB, (XB @ v).topk(10).indices) for v in V]
p_rand = [purity(TA, torch.randperm(10000)[:10])[0] for _ in range(1000)]
R["illusion1"] = {"mean_purity_A": float(np.mean([p[0] for p in pA])), "pure_A": sum(p[0] == 1 for p in pA),
                  "mean_purity_B": float(np.mean([p[0] for p in pB])), "pure_B": sum(p[0] == 1 for p in pB),
                  "mean_purity_random_A": float(np.mean(p_rand)), "first_direction_A": pA[0][1:], "first_direction_B": pB[0][1:]}
print(f"top 10 along v, corpus A: mean purity {R['illusion1']['mean_purity_A']:.3f}, directions with purity 1.0: {R['illusion1']['pure_A']} of 100")
print(f"top 10 along v, corpus B: mean purity {R['illusion1']['mean_purity_B']:.3f}, directions with purity 1.0: {R['illusion1']['pure_B']} of 100")
print(f"10 random points of corpus A, 1000 draws: mean purity {R['illusion1']['mean_purity_random_A']:.3f}")
print(f"first direction: corpus A — {pA[0][2]} of 10 from topic {pA[0][1]}, corpus B — {pB[0][2]} of 10 from topic {pB[0][1]}")

**Exercise 2. What is the mean purity of the top 10 points over 100 directions in corpus A? Give the answer with three decimals.**

Inside each corpus the direction looks like a detector of one topic, although the topics here are clusters and no topic grows along $v$ over the whole space.

**Part 2, as in Friedman et al.** A query $q$ and 16 keys $k_i$ in $\mathbb R^8$, score $s_i = q^\top k_i$, attention pattern $\mathrm{softmax}(s)$, prediction: the key with the largest score. On ID data (from the distribution where the surrogate is fitted) coordinates 0–2 have standard deviation 1 and coordinates 3–7 have 0.05; on OOD data (outside that distribution) all coordinates have 1. The surrogate projects $q$ and $k_i$ onto the first 3 right singular vectors of the matrix whose rows are the queries and keys of 1000 ID sequences. Evaluation uses 2000 sequences of each kind: the share of matching predictions and the mean Jensen-Shannon divergence (JSD, in nats) between the attention patterns of the model and the surrogate. Seed 0 continues the generator of part 1.

In [ ]:
def sequences(n, in_distribution):
    sd = torch.ones(8)
    if in_distribution: sd[3:] = 0.05
    return torch.randn(n, 17, 8) * sd  # position 0 is the query, 1..16 are the keys

S = sequences(1000, True).reshape(-1, 8)
_, sing, Vt = torch.linalg.svd(S, full_matrices=False)
proj = Vt[:3].T @ Vt[:3]

def evaluate(Z):
    q, k = Z[:, 0], Z[:, 1:]
    s = torch.einsum("nd,nkd->nk", q, k); s_sur = torch.einsum("nd,nkd->nk", q @ proj, k @ proj)
    a, b = s.softmax(1), s_sur.softmax(1); m = (a + b) / 2
    kl = lambda p, r: (p * (p / r).log()).sum(1)
    return (s.argmax(1) == s_sur.argmax(1)).float().mean().item(), (0.5 * kl(a, m) + 0.5 * kl(b, m)).mean().item()

R["illusion2"] = {"singular_values": sing.tolist(), "ID": evaluate(sequences(2000, True)), "OOD": evaluate(sequences(2000, False))}
print("singular values of the ID matrix:", [round(v, 1) for v in R["illusion2"]["singular_values"]])
for kind in ["ID", "OOD"]:
    agree, jsd = R["illusion2"][kind]
    print(f"{kind:>3}: share of matching predictions {agree:.3f}, mean JSD {jsd:.3g}")

**Exercise 3. What share of the surrogate's predictions matches the model's predictions on OOD data? Give the answer with three decimals.**

The SVD is computed from data, so the surrogate describes the model where it was fitted: the discarded directions barely vary on ID data, while on OOD data they carry 5 of the 8 coordinates with equal spread.

## **3. Accuracy and online code length on the real and the control task**

**Data.** 600 word types, activations of dimension $d = 48$, 20,000 occurrences, with equiprobable types. Each type has its own random signature $2.0\cdot\mathcal N(0, I)$; the property of the type (0 or 1) is encoded weakly, as a shift along a single unit direction $w_{\mathrm{prop}}$:

$$h = \mathrm{sig}[\text{type}] + 0.3\cdot\mathcal N(0, I) + 1.6\,(2y - 1)\,w_{\mathrm{prop}}.$$

**Tasks.** The real task is the property label of the type $y$; in the control task each type gets a random label 0 or 1, as in Hewitt and Liang.

**Probes.** Logistic regression and an MLP with two hidden layers of 256 neurons. Both are trained with Adam, lr $10^{-3}$, 1500 steps on the whole training part; the weights are chosen by the lowest loss on 10% of the training part held out for stopping, like the dev-set stopping rule of Voita and Titov.

**Online code.** Blocks at fractions 0.1, 0.2, 0.4, 0.8, 1.6, 3.2, 6.25, 12.5, 25, 50 and 100% of the data, as in Voita and Titov. The first block is sent with a uniform code, 1 bit per label; every next block is encoded by a probe trained on all preceding occurrences and costs its summed cross-entropy on the block, in bits. The uniform code of all labels is 20,000 bits, and compression is 20,000 divided by the online code length. Accuracy is that of the probe trained on the first half of the occurrences, measured on the second half. Seeds 0–4: the seed sets both the data and the probe initialization. About 90 seconds per seed on three CPU threads.

In [ ]:
N_TYPES, DIM, N_OCC, HIDDEN, STEPS = 600, 48, 20000, 256, 1500
FRACTIONS = [0.1, 0.2, 0.4, 0.8, 1.6, 3.2, 6.25, 12.5, 25, 50, 100]

def make_data(seed):
    g = torch.Generator().manual_seed(seed)
    sig = 2.0 * torch.randn(N_TYPES, DIM, generator=g)
    w_prop = torch.randn(DIM, generator=g); w_prop /= w_prop.norm()
    prop = torch.randint(0, 2, (N_TYPES,), generator=g)
    control = torch.randint(0, 2, (N_TYPES,), generator=g)
    token = torch.randint(0, N_TYPES, (N_OCC,), generator=g)
    H = sig[token] + 0.3 * torch.randn(N_OCC, DIM, generator=g) + 1.6 * (2 * prop[token] - 1).float()[:, None] * w_prop
    return H, prop[token], control[token]

def make_probe(kind):
    if kind == "linear": return torch.nn.Linear(DIM, 2)
    return torch.nn.Sequential(torch.nn.Linear(DIM, HIDDEN), torch.nn.ReLU(), torch.nn.Linear(HIDDEN, HIDDEN), torch.nn.ReLU(), torch.nn.Linear(HIDDEN, 2))

def train(kind, X, y):
    model = make_probe(kind); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    n_stop = max(1, len(X) // 10)
    X_stop, y_stop, X_fit, y_fit = X[:n_stop], y[:n_stop], X[n_stop:], y[n_stop:]
    best, best_state = float("inf"), None
    for _ in range(STEPS):
        opt.zero_grad(); F.cross_entropy(model(X_fit), y_fit).backward(); opt.step()
        with torch.no_grad(): stop_loss = F.cross_entropy(model(X_stop), y_stop).item()
        if stop_loss < best: best, best_state = stop_loss, {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model

def online_code(kind, X, y):
    edges = [round(N_OCC * f / 100) for f in FRACTIONS]
    bits, half_accuracy = float(edges[0]), None  # first block: uniform code, log2(2) = 1 bit per label
    for start, end in zip(edges[:-1], edges[1:]):
        torch.manual_seed(start)
        model = train(kind, X[:start], y[:start])
        with torch.no_grad():
            bits += F.cross_entropy(model(X[start:end]), y[start:end], reduction="sum").item() / math.log(2)
            if start == N_OCC // 2: half_accuracy = (model(X[start:]).argmax(1) == y[start:]).float().mean().item()
    return half_accuracy, bits

R["mdl"] = {}
for seed in range(5):
    H, y_real, y_control = make_data(seed)
    for kind in ["linear", "mlp"]:
        for task, labels in [("real", y_real), ("control", y_control)]:
            acc, bits = online_code(kind, H, labels)
            R["mdl"].setdefault(f"{kind}/{task}", []).append({"seed": seed, "accuracy": acc, "bits": bits})
    print(f"seed {seed} done, {round(time.time() - T0)} s")

print(f"{'probe':<7} {'task':<8} {'accuracy':>9} {'online code, bits':>18} {'sd over seeds':>14} {'compression':>12}")
for key, rows in R["mdl"].items():
    kind, task = key.split("/")
    acc = np.mean([r["accuracy"] for r in rows]); bits = np.array([r["bits"] for r in rows])
    print(f"{kind:<7} {task:<8} {acc:>9.3f} {bits.mean():>18.0f} {bits.std(ddof=1):>14.0f} {N_OCC / bits.mean():>12.2f}")

**Exercise 4. By what factor is the online code of the MLP probe on the control task longer than on the real task, using the means over five seeds? Give the answer with two decimals.**

By accuracy the MLP probe does not tell the real task from the control one: with 20,000 occurrences of 600 types every type appears in the training half, and the MLP only needs to recognize the type by its signature and recall its label. The real label can be predicted from the direction $w_{\mathrm{prop}}$ for a type the probe has not seen yet, so it is cheap to encode in the early blocks; the control label of a new type cannot be guessed.

In [ ]:
mlp_real = np.mean([r["bits"] for r in R["mdl"]["mlp/real"]]); mlp_control = np.mean([r["bits"] for r in R["mdl"]["mlp/control"]])
print(f"MLP probe: control / real code length = {mlp_control:.0f} / {mlp_real:.0f} = {mlp_control / mlp_real:.2f}")
json.dump(R, open("toys_32_results.json", "w"))
print("saved toys_32_results.json; total time", round(time.time() - T0), "s")

## **Summary**

* The number $R(H)$ separates an incomplete hypothesis from a complete one where accuracy after the replacements is the same, and it depends on the behavior metric: chance level in accuracy does not coincide with the zero of the $R$ scale.
* A direction found from top examples inside one corpus and a surrogate fitted on one distribution are faithful where they were found and do not transfer beyond it.
* Equal probe accuracy on the real and the control task does not mean equal information in the activations: the online code length tells them apart.